# KV Cache 专项 Notebook：Paged 分配 + LRU 驱逐 + 命中率统计 + 可视化

这个 notebook 目标是把 KV Cache 核心机制跑通并可观测：

1. `Paged` 块分配（固定 block 容量）
2. `LRU` 驱逐（按前缀缓存条目淘汰）
3. 命中率 / 驱逐次数 / 块利用率统计
4. 可视化趋势图（命中率、占用、驱逐）

> 说明：这是面试和学习用途的简化模拟器，不追求底层 kernel 细节。

In [ ]:
import math
from dataclasses import dataclass, field
from typing import Dict, List

import numpy as np

# 可视化依赖（若环境没有 matplotlib，会自动降级为文本输出）
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

np.random.seed(42)
print("Ready. NumPy loaded.", "matplotlib:", HAS_MPL)

## 1) 定义核心数据结构

- `PrefixEntry`：表示一个可复用前缀在缓存中的元数据
- `PagedPrefixKVCache`：核心缓存模拟器，支持
  - 固定大小 block 池
  - 缺块时 LRU 驱逐
  - 请求级统计和时间序列记录

In [ ]:
@dataclass
class PrefixEntry:
    key: str
    blocks: List[int]
    token_len: int
    last_access_step: int
    use_count: int = 0


class PagedPrefixKVCache:
    def __init__(self, num_blocks: int = 512, block_size_tokens: int = 16):
        if num_blocks <= 0 or block_size_tokens <= 0:
            raise ValueError("num_blocks and block_size_tokens must be positive")

        self.num_blocks = num_blocks
        self.block_size_tokens = block_size_tokens
        self.free_blocks: List[int] = list(range(num_blocks))
        self.entries: Dict[str, PrefixEntry] = {}

        # 全局计数
        self.step = 0
        self.total_requests = 0
        self.hits = 0
        self.misses = 0
        self.evictions = 0
        self.evicted_blocks = 0
        self.oom_events = 0

        # 统计与时间序列
        self.prefix_stats: Dict[str, Dict[str, int]] = {}
        self.history: List[Dict[str, float]] = []

    def _required_blocks(self, tokens: int) -> int:
        if tokens <= 0:
            return 0
        return int(math.ceil(tokens / self.block_size_tokens))

    def _ensure_prefix_stat(self, prefix_key: str) -> None:
        if prefix_key not in self.prefix_stats:
            self.prefix_stats[prefix_key] = {
                "access": 0,
                "hit": 0,
                "miss": 0,
                "evicted": 0,
            }

    def _evict_lru_one(self):
        if not self.entries:
            raise RuntimeError("No cache entry available to evict")

        # LRU: last_access_step 最小者优先淘汰
        lru_key = min(self.entries.keys(), key=lambda k: (self.entries[k].last_access_step, k))
        entry = self.entries.pop(lru_key)
        self.free_blocks.extend(entry.blocks)

        self.evictions += 1
        self.evicted_blocks += len(entry.blocks)
        self._ensure_prefix_stat(lru_key)
        self.prefix_stats[lru_key]["evicted"] += 1

        return lru_key, len(entry.blocks)

    def _allocate_blocks(self, num_needed: int):
        if num_needed < 0:
            raise ValueError("num_needed must be >= 0")
        if num_needed == 0:
            return [], []
        if num_needed > self.num_blocks:
            raise ValueError("Requested blocks exceed cache capacity")

        evicted = []
        while len(self.free_blocks) < num_needed:
            if not self.entries:
                self.oom_events += 1
                raise RuntimeError("Insufficient free blocks and nothing to evict")
            evicted.append(self._evict_lru_one())

        blocks = [self.free_blocks.pop() for _ in range(num_needed)]
        return blocks, evicted

    def _record_history(self, is_hit: bool, evicted_this_req: int) -> None:
        used_blocks = self.num_blocks - len(self.free_blocks)
        hit_rate = self.hits / max(1, self.total_requests)
        self.history.append(
            {
                "step": self.step,
                "hit": 1 if is_hit else 0,
                "cum_hit_rate": hit_rate,
                "used_blocks": used_blocks,
                "free_blocks": len(self.free_blocks),
                "evictions_this_req": evicted_this_req,
                "cum_evictions": self.evictions,
            }
        )

    def access_request(self, prefix_key: str, prefix_tokens: int, decode_tokens: int = 0):
        self.step += 1
        self.total_requests += 1
        self._ensure_prefix_stat(prefix_key)
        self.prefix_stats[prefix_key]["access"] += 1

        prefix_blocks = self._required_blocks(prefix_tokens)
        decode_blocks = self._required_blocks(decode_tokens)

        evicted_this_req = 0
        is_hit = False

        # 1) 访问/填充前缀缓存
        if prefix_key in self.entries and self.entries[prefix_key].token_len == prefix_tokens:
            is_hit = True
            self.hits += 1
            self.prefix_stats[prefix_key]["hit"] += 1
            entry = self.entries[prefix_key]
            entry.last_access_step = self.step
            entry.use_count += 1
        else:
            self.misses += 1
            self.prefix_stats[prefix_key]["miss"] += 1

            # 同 key 但长度变化时，旧前缀作废
            if prefix_key in self.entries:
                stale = self.entries.pop(prefix_key)
                self.free_blocks.extend(stale.blocks)

            new_blocks, evicted = self._allocate_blocks(prefix_blocks)
            evicted_this_req += len(evicted)
            self.entries[prefix_key] = PrefixEntry(
                key=prefix_key,
                blocks=new_blocks,
                token_len=prefix_tokens,
                last_access_step=self.step,
                use_count=1,
            )

        # 2) 为 decode 申请临时块（模拟 decode 对容量压力），请求结束后立即释放
        temp_blocks = []
        if decode_blocks > 0:
            temp_blocks, evicted = self._allocate_blocks(decode_blocks)
            evicted_this_req += len(evicted)
            self.free_blocks.extend(temp_blocks)

        self._record_history(is_hit=is_hit, evicted_this_req=evicted_this_req)

        return {
            "hit": is_hit,
            "evictions_this_request": evicted_this_req,
            "resident_prefix_entries": len(self.entries),
            "used_blocks": self.num_blocks - len(self.free_blocks),
        }

    def summary(self) -> Dict[str, float]:
        used_ratio_series = [h["used_blocks"] / self.num_blocks for h in self.history] if self.history else [0.0]
        return {
            "capacity_blocks": self.num_blocks,
            "block_size_tokens": self.block_size_tokens,
            "total_requests": self.total_requests,
            "hit_rate": self.hits / max(1, self.total_requests),
            "miss_rate": self.misses / max(1, self.total_requests),
            "evictions": self.evictions,
            "evicted_blocks": self.evicted_blocks,
            "resident_prefix_entries": len(self.entries),
            "avg_block_utilization": float(np.mean(used_ratio_series)),
            "p95_block_utilization": float(np.percentile(used_ratio_series, 95)),
            "oom_events": self.oom_events,
        }

## 2) 生成 Zipf 工作负载

我们用 Zipf 分布模拟“热点前缀”：少数 prefix 被频繁访问，多数 prefix 长尾访问。

In [ ]:
def generate_zipf_workload(
    n_requests: int = 600,
    n_prefixes: int = 80,
    zipf_s: float = 1.25,
    min_prefix_tokens: int = 128,
    max_prefix_tokens: int = 4096,
    min_decode_tokens: int = 16,
    max_decode_tokens: int = 256,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)

    prefix_keys = [f"prefix_{i:03d}" for i in range(n_prefixes)]
    ranks = np.arange(1, n_prefixes + 1)
    probs = 1 / np.power(ranks, zipf_s)
    probs = probs / probs.sum()

    # 每个 prefix 固定一个 token 长度，便于复用命中
    prefix_token_table = {
        k: int(rng.integers(min_prefix_tokens, max_prefix_tokens + 1))
        for k in prefix_keys
    }

    workload = []
    for i in range(n_requests):
        key = rng.choice(prefix_keys, p=probs)
        workload.append(
            {
                "req_id": i,
                "prefix_key": key,
                "prefix_tokens": prefix_token_table[key],
                "decode_tokens": int(rng.integers(min_decode_tokens, max_decode_tokens + 1)),
            }
        )

    return workload


workload = generate_zipf_workload()
print("workload size:", len(workload), "sample:", workload[:2])

## 3) 运行模拟并输出核心指标

In [ ]:
def run_simulation(
    workload,
    num_blocks: int = 320,
    block_size_tokens: int = 16,
):
    cache = PagedPrefixKVCache(num_blocks=num_blocks, block_size_tokens=block_size_tokens)
    traces = []
    for req in workload:
        out = cache.access_request(
            prefix_key=req["prefix_key"],
            prefix_tokens=req["prefix_tokens"],
            decode_tokens=req["decode_tokens"],
        )
        traces.append(out)
    return cache, traces


cache, traces = run_simulation(workload, num_blocks=320, block_size_tokens=16)
summary = cache.summary()

print("=== Simulation Summary ===")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"{k:>24}: {v:.4f}")
    else:
        print(f"{k:>24}: {v}")

print("\nRecent 5 request traces:")
for t in traces[-5:]:
    print(t)

## 4) 查看热点前缀统计（命中 / 未命中 / 被驱逐）

In [ ]:
def top_prefix_stats(cache: PagedPrefixKVCache, top_k: int = 12):
    rows = []
    for key, s in cache.prefix_stats.items():
        access = s["access"]
        hit = s["hit"]
        miss = s["miss"]
        evicted = s["evicted"]
        hit_rate = hit / max(1, access)
        rows.append((key, access, hit, miss, evicted, hit_rate))

    rows.sort(key=lambda x: x[1], reverse=True)
    return rows[:top_k]


rows = top_prefix_stats(cache, top_k=12)
print(f"{'prefix':<12} {'access':>7} {'hit':>6} {'miss':>6} {'evicted':>8} {'hit_rate':>9}")
for r in rows:
    print(f"{r[0]:<12} {r[1]:>7d} {r[2]:>6d} {r[3]:>6d} {r[4]:>8d} {r[5]:>9.2%}")

## 5) 可视化：命中率、块占用、驱逐趋势

In [ ]:
hist = cache.history
steps = np.array([h["step"] for h in hist])
cum_hit_rate = np.array([h["cum_hit_rate"] for h in hist])
used_blocks = np.array([h["used_blocks"] for h in hist])
cum_evictions = np.array([h["cum_evictions"] for h in hist])
evictions_each = np.array([h["evictions_this_req"] for h in hist])

if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].plot(steps, cum_hit_rate)
    axes[0, 0].set_title("Cumulative Hit Rate")
    axes[0, 0].set_xlabel("request step")
    axes[0, 0].set_ylabel("hit rate")
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(steps, used_blocks, label="used")
    axes[0, 1].plot(steps, cache.num_blocks - used_blocks, label="free")
    axes[0, 1].set_title("Block Usage")
    axes[0, 1].set_xlabel("request step")
    axes[0, 1].set_ylabel("blocks")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(steps, cum_evictions)
    axes[1, 0].set_title("Cumulative Evictions")
    axes[1, 0].set_xlabel("request step")
    axes[1, 0].set_ylabel("count")
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].bar(np.arange(len(evictions_each)), evictions_each)
    axes[1, 1].set_title("Evictions Per Request")
    axes[1, 1].set_xlabel("request step")
    axes[1, 1].set_ylabel("evictions in request")

    plt.tight_layout()
    plt.show()
else:
    print("matplotlib 不可用，输出文本摘要：")
    print("final cumulative hit rate:", float(cum_hit_rate[-1]))
    print("max used blocks:", int(np.max(used_blocks)))
    print("total evictions:", int(cum_evictions[-1]))

## 6) 容量敏感性实验（可选但强烈推荐）

对比不同 `num_blocks` 下的 hit rate 与 eviction，帮助你形成容量规划直觉。

In [ ]:
# 先根据 workload 计算“最小可行容量”（至少容纳一个最大前缀）
min_feasible_capacity = max(int(math.ceil(req["prefix_tokens"] / 16)) for req in workload)
capacities = [
    min_feasible_capacity,
    min_feasible_capacity + 40,
    min_feasible_capacity + 80,
    min_feasible_capacity + 140,
    min_feasible_capacity + 220,
]

records = []
for cap in capacities:
    c, _ = run_simulation(workload, num_blocks=cap, block_size_tokens=16)
    s = c.summary()
    records.append((cap, s["hit_rate"], s["evictions"], s["avg_block_utilization"]))

print("min feasible capacity:", min_feasible_capacity)
print(f"{'capacity':>10} {'hit_rate':>10} {'evictions':>10} {'avg_util':>10}")
for cap, hr, ev, util in records:
    print(f"{cap:>10d} {hr:>10.2%} {ev:>10.0f} {util:>10.2%}")

if HAS_MPL:
    x = np.array([r[0] for r in records])
    y_hit = np.array([r[1] for r in records])
    y_evict = np.array([r[2] for r in records])

    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    ax1.plot(x, y_hit, marker="o", label="hit rate")
    ax1.set_xlabel("capacity (blocks)")
    ax1.set_ylabel("hit rate")
    ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(x, y_evict, marker="s", color="tab:red", label="evictions")
    ax2.set_ylabel("evictions")

    ax1.set_title("Capacity vs Hit Rate / Evictions")
    plt.tight_layout()
    plt.show()

## 7) 你可以重点观察什么

- 当容量不足时：`evictions` 增长更快，`hit_rate` 明显下降。
- 热点前缀通常有更高命中率；长尾前缀更容易被淘汰。
- decode 临时块会挤压前缀缓存空间，导致额外驱逐。

### 建议你接下来自己改的参数

1. `zipf_s`（热点集中度）
2. `n_prefixes`（长尾规模）
3. `max_decode_tokens`（decode 压力）
4. `num_blocks`（缓存容量）

> 面试表达模板：
> “我会先通过 workload 分布估计热点前缀容量，再用 hit rate/eviction 曲线找拐点，最后结合 SLO 选容量并预留突发缓冲。”

## 8) 驱逐策略对比：LRU vs LFU

下面扩展一个可切换策略的缓存类，保持其余逻辑一致，只替换“挑选 victim”规则。

In [ ]:
class PolicyPagedPrefixKVCache(PagedPrefixKVCache):
    def __init__(self, num_blocks: int = 512, block_size_tokens: int = 16, eviction_policy: str = "lru"):
        super().__init__(num_blocks=num_blocks, block_size_tokens=block_size_tokens)
        policy = eviction_policy.lower().strip()
        if policy not in {"lru", "lfu"}:
            raise ValueError("eviction_policy must be one of {'lru', 'lfu'}")
        self.eviction_policy = policy

    def _evict_lru_one(self):
        if not self.entries:
            raise RuntimeError("No cache entry available to evict")

        if self.eviction_policy == "lru":
            victim_key = min(
                self.entries.keys(),
                key=lambda k: (self.entries[k].last_access_step, k),
            )
        else:  # LFU with LRU tie-break
            victim_key = min(
                self.entries.keys(),
                key=lambda k: (self.entries[k].use_count, self.entries[k].last_access_step, k),
            )

        entry = self.entries.pop(victim_key)
        self.free_blocks.extend(entry.blocks)

        self.evictions += 1
        self.evicted_blocks += len(entry.blocks)
        self._ensure_prefix_stat(victim_key)
        self.prefix_stats[victim_key]["evicted"] += 1

        return victim_key, len(entry.blocks)


def run_policy_experiment(workload, policy: str, num_blocks: int = 320, block_size_tokens: int = 16):
    cache = PolicyPagedPrefixKVCache(
        num_blocks=num_blocks,
        block_size_tokens=block_size_tokens,
        eviction_policy=policy,
    )
    for req in workload:
        cache.access_request(
            prefix_key=req["prefix_key"],
            prefix_tokens=req["prefix_tokens"],
            decode_tokens=req["decode_tokens"],
        )
    return cache.summary(), cache

In [ ]:
compare_capacity = 320
summary_lru, cache_lru = run_policy_experiment(workload, policy="lru", num_blocks=compare_capacity)
summary_lfu, cache_lfu = run_policy_experiment(workload, policy="lfu", num_blocks=compare_capacity)

print(f"capacity={compare_capacity} blocks")
print(f"{'policy':<8} {'hit_rate':>10} {'evictions':>10} {'avg_util':>10} {'p95_util':>10}")
for name, s in [("lru", summary_lru), ("lfu", summary_lfu)]:
    print(
        f"{name:<8} {s['hit_rate']:>10.2%} {s['evictions']:>10.0f} "
        f"{s['avg_block_utilization']:>10.2%} {s['p95_block_utilization']:>10.2%}"
    )

if HAS_MPL:
    policies = ["LRU", "LFU"]
    hit_rates = [summary_lru["hit_rate"], summary_lfu["hit_rate"]]
    evictions = [summary_lru["evictions"], summary_lfu["evictions"]]

    fig, ax1 = plt.subplots(figsize=(7.5, 4.5))
    x = np.arange(len(policies))
    w = 0.35

    ax1.bar(x - w / 2, hit_rates, width=w, label="hit rate")
    ax1.set_ylabel("hit rate")
    ax1.set_xticks(x)
    ax1.set_xticklabels(policies)
    ax1.set_ylim(0, max(hit_rates) * 1.3 if max(hit_rates) > 0 else 1)

    ax2 = ax1.twinx()
    ax2.bar(x + w / 2, evictions, width=w, color="tab:red", label="evictions")
    ax2.set_ylabel("evictions")

    ax1.set_title("LRU vs LFU")
    plt.tight_layout()
    plt.show()

## 9) 多租户公平驱逐（Weighted Fair Eviction）

场景：一个大租户流量很高时，普通 LRU 可能把小租户工作集不断挤掉。

这里实现一个简化公平驱逐：
- 每个租户有目标权重（`tenant_weights`）
- 驱逐时优先从“超配份额”的租户中挑最旧条目

In [ ]:
class MultiTenantFairKVCache(PagedPrefixKVCache):
    def __init__(
        self,
        num_blocks: int = 512,
        block_size_tokens: int = 16,
        tenant_weights: Dict[str, float] = None,
    ):
        super().__init__(num_blocks=num_blocks, block_size_tokens=block_size_tokens)
        self.tenant_weights = tenant_weights or {}

    @staticmethod
    def _tenant_of_key(prefix_key: str) -> str:
        return prefix_key.split("::", 1)[0] if "::" in prefix_key else "default"

    def _tenant_usage_blocks(self) -> Dict[str, int]:
        usage = {}
        for entry in self.entries.values():
            t = self._tenant_of_key(entry.key)
            usage[t] = usage.get(t, 0) + len(entry.blocks)
        return usage

    def _normalized_tenant_weights(self) -> Dict[str, float]:
        tenants = set(self.tenant_weights.keys())
        for k in self.entries:
            tenants.add(self._tenant_of_key(k))
        if not tenants:
            return {"default": 1.0}

        weights = {t: float(self.tenant_weights.get(t, 1.0)) for t in tenants}
        total = sum(weights.values())
        return {t: w / total for t, w in weights.items()}

    def _evict_lru_one(self):
        if not self.entries:
            raise RuntimeError("No cache entry available to evict")

        usage = self._tenant_usage_blocks()
        target = self._normalized_tenant_weights()
        quotas = {t: target.get(t, 0.0) * self.num_blocks for t in usage}

        # 先找超配租户（占用超过目标配额）。
        over_tenants = [t for t in usage if usage[t] > quotas.get(t, 0.0)]
        if over_tenants:
            victim_tenant = max(
                over_tenants,
                key=lambda t: (usage[t] - quotas.get(t, 0.0), t),
            )
            candidate_keys = [k for k in self.entries if self._tenant_of_key(k) == victim_tenant]
            victim_key = min(
                candidate_keys,
                key=lambda k: (self.entries[k].last_access_step, k),
            )
        else:
            # 若无人超配，则退化为全局 LRU。
            victim_key = min(
                self.entries.keys(),
                key=lambda k: (self.entries[k].last_access_step, k),
            )

        entry = self.entries.pop(victim_key)
        self.free_blocks.extend(entry.blocks)

        self.evictions += 1
        self.evicted_blocks += len(entry.blocks)
        self._ensure_prefix_stat(victim_key)
        self.prefix_stats[victim_key]["evicted"] += 1

        return victim_key, len(entry.blocks)


def generate_multi_tenant_workload(
    n_requests: int = 900,
    zipf_s: float = 1.2,
    block_size_tokens: int = 16,
    seed: int = 7,
):
    rng = np.random.default_rng(seed)

    # tenant: (request_share, num_prefixes)
    tenant_cfg = {
        "A": (0.68, 60),
        "B": (0.22, 24),
        "C": (0.10, 16),
    }

    tenants = list(tenant_cfg.keys())
    tenant_probs = np.array([tenant_cfg[t][0] for t in tenants], dtype=float)
    tenant_probs = tenant_probs / tenant_probs.sum()

    tenant_prefixes = {}
    tenant_prefix_probs = {}
    prefix_token_table = {}

    for t in tenants:
        _, n_prefix = tenant_cfg[t]
        keys = [f"{t}::prefix_{i:03d}" for i in range(n_prefix)]
        tenant_prefixes[t] = keys

        ranks = np.arange(1, n_prefix + 1)
        p = 1 / np.power(ranks, zipf_s)
        p = p / p.sum()
        tenant_prefix_probs[t] = p

        for k in keys:
            # 保证至少 1 block
            tok = int(rng.integers(block_size_tokens, 4096 + 1))
            prefix_token_table[k] = tok

    workload = []
    for i in range(n_requests):
        t = rng.choice(tenants, p=tenant_probs)
        key = rng.choice(tenant_prefixes[t], p=tenant_prefix_probs[t])
        workload.append(
            {
                "req_id": i,
                "tenant": t,
                "prefix_key": key,
                "prefix_tokens": prefix_token_table[key],
                "decode_tokens": int(rng.integers(16, 256 + 1)),
            }
        )

    return workload


def run_tenant_workload(workload, cache):
    tenant_stats = {}
    for req in workload:
        t = req["tenant"]
        if t not in tenant_stats:
            tenant_stats[t] = {"access": 0, "hit": 0}
        tenant_stats[t]["access"] += 1

        out = cache.access_request(
            prefix_key=req["prefix_key"],
            prefix_tokens=req["prefix_tokens"],
            decode_tokens=req["decode_tokens"],
        )
        if out["hit"]:
            tenant_stats[t]["hit"] += 1

    hit_rates = {t: tenant_stats[t]["hit"] / max(1, tenant_stats[t]["access"]) for t in tenant_stats}
    return cache.summary(), tenant_stats, hit_rates


def jains_fairness(values: List[float]) -> float:
    arr = np.array(values, dtype=float)
    if arr.size == 0:
        return 0.0
    denom = arr.size * np.sum(arr ** 2)
    if denom == 0:
        return 0.0
    return float((np.sum(arr) ** 2) / denom)

In [ ]:
tenant_workload = generate_multi_tenant_workload(n_requests=900, zipf_s=1.2, seed=7)

cap = 340
base_cache = PolicyPagedPrefixKVCache(num_blocks=cap, block_size_tokens=16, eviction_policy="lru")
fair_cache = MultiTenantFairKVCache(
    num_blocks=cap,
    block_size_tokens=16,
    tenant_weights={"A": 1.0, "B": 1.0, "C": 1.0},  # 目标：租户等权
)

sum_base, stat_base, hr_base = run_tenant_workload(tenant_workload, base_cache)
sum_fair, stat_fair, hr_fair = run_tenant_workload(tenant_workload, fair_cache)

print(f"capacity={cap} blocks")
print(f"{'metric':<20} {'LRU':>12} {'Fair':>12}")
print(f"{'global hit rate':<20} {sum_base['hit_rate']:>12.2%} {sum_fair['hit_rate']:>12.2%}")
print(f"{'evictions':<20} {sum_base['evictions']:>12.0f} {sum_fair['evictions']:>12.0f}")
print(
    f"{'Jain fairness(hit)':<20} "
    f"{jains_fairness(list(hr_base.values())):>12.4f} "
    f"{jains_fairness(list(hr_fair.values())):>12.4f}"
)

print("\nPer-tenant hit rate:")
for t in sorted(hr_base.keys()):
    print(f"tenant={t}  LRU={hr_base[t]:.2%}  Fair={hr_fair[t]:.2%}")

if HAS_MPL:
    tenants = sorted(hr_base.keys())
    x = np.arange(len(tenants))
    w = 0.35

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.bar(x - w / 2, [hr_base[t] for t in tenants], width=w, label="LRU")
    ax.bar(x + w / 2, [hr_fair[t] for t in tenants], width=w, label="Fair")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Tenant {t}" for t in tenants])
    ax.set_ylabel("hit rate")
    ax.set_title("Per-tenant Hit Rate: LRU vs Fair Eviction")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10) 面试可讲的结论模板

- **策略对比**：LRU 简单稳健；LFU 在“稳定热点”场景命中率常更高，但对热点漂移更敏感。
- **公平性**：仅看全局 hit rate 可能掩盖租户不公平，建议同时看 per-tenant hit rate + Jain 指数。
- **工程建议**：线上通常使用“多信号混合驱逐”（recency + frequency + tenant budget + priority）。
- **回滚条件**：若某租户 P99 或 hit rate 跌破阈值，触发策略降级（回到 LRU/提高该租户预算）。

## 11) Adaptive 策略：基于滑动窗口自动切换 LRU / LFU / Fair

核心思路：
- 维护一个滑动窗口（最近 W 次请求）的 **hit rate** 和 **Jain fairness index**
- 每个周期评估当前策略表现，若低于阈值则切换到预期更好的策略
- 决策规则（简化版）：
  1. 若窗口 hit rate < `hit_threshold` → 切到 LFU（强化频率信号）
  2. 若窗口 Jain index < `fair_threshold` → 切到 Fair（保护小租户）
  3. 否则保持当前策略（默认 LRU）
- 带冷却期（`cooldown`），避免频繁抖动

In [ ]:
from collections import deque

POLICY_LRU = "lru"
POLICY_LFU = "lfu"
POLICY_FAIR = "fair"


class AdaptiveKVCache(PagedPrefixKVCache):
    """
    自适应驱逐策略：基于滑动窗口 hit rate 与 Jain fairness 动态切换。
    """

    def __init__(
        self,
        num_blocks: int = 512,
        block_size_tokens: int = 16,
        tenant_weights: Dict[str, float] = None,
        window_size: int = 60,
        eval_interval: int = 30,
        hit_threshold: float = 0.20,
        fair_threshold: float = 0.85,
        cooldown: int = 40,
    ):
        super().__init__(num_blocks=num_blocks, block_size_tokens=block_size_tokens)
        self.tenant_weights = tenant_weights or {}
        self.current_policy = POLICY_LRU

        # 窗口与评估参数
        self.window_size = window_size
        self.eval_interval = eval_interval
        self.hit_threshold = hit_threshold
        self.fair_threshold = fair_threshold
        self.cooldown = cooldown

        # 滑动窗口
        self._hit_window: deque = deque(maxlen=window_size)
        self._tenant_hit_window: Dict[str, deque] = {}

        # 冷却计数
        self._cooldown_remaining = 0

        # 切换日志
        self.switch_log: List[Dict] = []

    # ---- 租户辅助 ----
    @staticmethod
    def _tenant_of_key(prefix_key: str) -> str:
        return prefix_key.split("::", 1)[0] if "::" in prefix_key else "default"

    def _tenant_usage_blocks(self) -> Dict[str, int]:
        usage: Dict[str, int] = {}
        for entry in self.entries.values():
            t = self._tenant_of_key(entry.key)
            usage[t] = usage.get(t, 0) + len(entry.blocks)
        return usage

    def _normalized_tenant_weights(self) -> Dict[str, float]:
        tenants = set(self.tenant_weights.keys())
        for k in self.entries:
            tenants.add(self._tenant_of_key(k))
        if not tenants:
            return {"default": 1.0}
        weights = {t: float(self.tenant_weights.get(t, 1.0)) for t in tenants}
        total = sum(weights.values())
        return {t: w / total for t, w in weights.items()}

    # ---- 驱逐核心：按 current_policy 分派 ----
    def _evict_lru_one(self):
        if not self.entries:
            raise RuntimeError("No cache entry available to evict")

        if self.current_policy == POLICY_LFU:
            victim_key = min(
                self.entries.keys(),
                key=lambda k: (self.entries[k].use_count, self.entries[k].last_access_step, k),
            )
        elif self.current_policy == POLICY_FAIR:
            victim_key = self._pick_fair_victim()
        else:  # LRU
            victim_key = min(
                self.entries.keys(),
                key=lambda k: (self.entries[k].last_access_step, k),
            )

        entry = self.entries.pop(victim_key)
        self.free_blocks.extend(entry.blocks)
        self.evictions += 1
        self.evicted_blocks += len(entry.blocks)
        self._ensure_prefix_stat(victim_key)
        self.prefix_stats[victim_key]["evicted"] += 1
        return victim_key, len(entry.blocks)

    def _pick_fair_victim(self) -> str:
        usage = self._tenant_usage_blocks()
        target = self._normalized_tenant_weights()
        quotas = {t: target.get(t, 0.0) * self.num_blocks for t in usage}
        over_tenants = [t for t in usage if usage[t] > quotas.get(t, 0.0)]
        if over_tenants:
            victim_tenant = max(over_tenants, key=lambda t: (usage[t] - quotas.get(t, 0.0), t))
            candidate_keys = [k for k in self.entries if self._tenant_of_key(k) == victim_tenant]
            return min(candidate_keys, key=lambda k: (self.entries[k].last_access_step, k))
        return min(self.entries.keys(), key=lambda k: (self.entries[k].last_access_step, k))

    # ---- 窗口统计 ----
    def _window_hit_rate(self) -> float:
        if not self._hit_window:
            return 0.0
        return sum(self._hit_window) / len(self._hit_window)

    def _window_jain_fairness(self) -> float:
        if not self._tenant_hit_window:
            return 1.0
        rates = []
        for t, dq in self._tenant_hit_window.items():
            if len(dq) > 0:
                rates.append(sum(dq) / len(dq))
        if not rates:
            return 1.0
        arr = np.array(rates)
        denom = len(arr) * np.sum(arr ** 2)
        if denom == 0:
            return 1.0
        return float((np.sum(arr) ** 2) / denom)

    # ---- 自适应评估 ----
    def _maybe_switch_policy(self):
        if self._cooldown_remaining > 0:
            self._cooldown_remaining -= 1
            return
        if self.step % self.eval_interval != 0:
            return
        if len(self._hit_window) < self.window_size // 2:
            return

        win_hr = self._window_hit_rate()
        win_jain = self._window_jain_fairness()
        old_policy = self.current_policy

        if win_jain < self.fair_threshold and self.current_policy != POLICY_FAIR:
            self.current_policy = POLICY_FAIR
        elif win_hr < self.hit_threshold and self.current_policy != POLICY_LFU:
            self.current_policy = POLICY_LFU
        elif win_hr >= self.hit_threshold and win_jain >= self.fair_threshold and self.current_policy != POLICY_LRU:
            self.current_policy = POLICY_LRU

        if self.current_policy != old_policy:
            self._cooldown_remaining = self.cooldown
            self.switch_log.append({
                "step": self.step,
                "from": old_policy,
                "to": self.current_policy,
                "win_hr": round(win_hr, 4),
                "win_jain": round(win_jain, 4),
            })

    # ---- 覆写 access_request ----
    def access_request(self, prefix_key: str, prefix_tokens: int, decode_tokens: int = 0):
        result = super().access_request(prefix_key, prefix_tokens, decode_tokens)

        # 更新滑动窗口
        self._hit_window.append(1 if result["hit"] else 0)
        tenant = self._tenant_of_key(prefix_key)
        if tenant not in self._tenant_hit_window:
            self._tenant_hit_window[tenant] = deque(maxlen=self.window_size)
        self._tenant_hit_window[tenant].append(1 if result["hit"] else 0)

        # 评估是否切换策略
        self._maybe_switch_policy()

        return result

In [ ]:
# 用多租户 workload 跑 Adaptive vs 静态策略对比
tenant_wl = generate_multi_tenant_workload(n_requests=900, zipf_s=1.2, seed=7)
cap = 340

# --- static LRU ---
static_lru = PolicyPagedPrefixKVCache(num_blocks=cap, block_size_tokens=16, eviction_policy="lru")
sum_lru, _, hr_lru = run_tenant_workload(tenant_wl, static_lru)

# --- static LFU ---
static_lfu = PolicyPagedPrefixKVCache(num_blocks=cap, block_size_tokens=16, eviction_policy="lfu")
sum_lfu, _, hr_lfu = run_tenant_workload(tenant_wl, static_lfu)

# --- static Fair ---
static_fair = MultiTenantFairKVCache(num_blocks=cap, block_size_tokens=16, tenant_weights={"A": 1, "B": 1, "C": 1})
sum_fair, _, hr_fair = run_tenant_workload(tenant_wl, static_fair)

# --- Adaptive ---
adaptive = AdaptiveKVCache(
    num_blocks=cap,
    block_size_tokens=16,
    tenant_weights={"A": 1, "B": 1, "C": 1},
    window_size=60,
    eval_interval=30,
    hit_threshold=0.15,
    fair_threshold=0.80,
    cooldown=40,
)
sum_adap, stat_adap, hr_adap = run_tenant_workload(tenant_wl, adaptive)

# 汇总
print(f"capacity={cap} blocks, requests={len(tenant_wl)}")
print(f"{'strategy':<12} {'hit_rate':>10} {'evictions':>10} {'jain(hit)':>10}")
for name, sm, hr in [
    ("LRU", sum_lru, hr_lru),
    ("LFU", sum_lfu, hr_lfu),
    ("Fair", sum_fair, hr_fair),
    ("Adaptive", sum_adap, hr_adap),
]:
    j = jains_fairness(list(hr.values()))
    print(f"{name:<12} {sm['hit_rate']:>10.2%} {sm['evictions']:>10.0f} {j:>10.4f}")

print(f"\nAdaptive per-tenant hit rate:")
for t in sorted(hr_adap.keys()):
    print(f"  tenant={t}  {hr_adap[t]:.2%}")

print(f"\nAdaptive switch log ({len(adaptive.switch_log)} switches):")
for s in adaptive.switch_log:
    print(f"  step={s['step']:>4d}  {s['from']:>5s} -> {s['to']:<5s}  win_hr={s['win_hr']:.3f}  win_jain={s['win_jain']:.3f}")

In [ ]:
# 可视化 Adaptive 策略的动态行为
if HAS_MPL:
    hist_a = adaptive.history
    steps_a = np.array([h["step"] for h in hist_a])
    cum_hr_a = np.array([h["cum_hit_rate"] for h in hist_a])
    used_a = np.array([h["used_blocks"] for h in hist_a])

    # 滑动窗口 hit rate（手动重算）
    win_hr_series = []
    win = deque(maxlen=adaptive.window_size)
    for h in hist_a:
        win.append(h["hit"])
        win_hr_series.append(sum(win) / len(win))
    win_hr_series = np.array(win_hr_series)

    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    # 1) 累计 vs 滑动窗口 hit rate
    axes[0].plot(steps_a, cum_hr_a, label="cumulative hit rate", alpha=0.7)
    axes[0].plot(steps_a, win_hr_series, label=f"window hit rate (W={adaptive.window_size})", alpha=0.9)
    axes[0].axhline(adaptive.hit_threshold, color="red", ls="--", lw=1, label=f"hit threshold={adaptive.hit_threshold}")
    for sw in adaptive.switch_log:
        axes[0].axvline(sw["step"], color="gray", ls=":", lw=0.8)
    axes[0].set_ylabel("hit rate")
    axes[0].legend(fontsize=8)
    axes[0].set_title("Adaptive KV Cache: Hit Rate & Policy Switches")
    axes[0].grid(alpha=0.3)

    # 2) Block 占用
    axes[1].plot(steps_a, used_a, color="tab:orange")
    axes[1].axhline(cap, color="red", ls="--", lw=1, label=f"capacity={cap}")
    for sw in adaptive.switch_log:
        axes[1].axvline(sw["step"], color="gray", ls=":", lw=0.8)
    axes[1].set_ylabel("used blocks")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    # 3) 策略时间线
    policy_map = {POLICY_LRU: 0, POLICY_LFU: 1, POLICY_FAIR: 2}
    policy_labels = ["LRU", "LFU", "Fair"]
    # 重建策略序列
    policy_seq = []
    sw_iter = iter(adaptive.switch_log)
    next_sw = next(sw_iter, None)
    cur_p = POLICY_LRU
    for s in steps_a:
        while next_sw and s >= next_sw["step"]:
            cur_p = next_sw["to"]
            next_sw = next(sw_iter, None)
        policy_seq.append(policy_map[cur_p])
    policy_seq = np.array(policy_seq)

    axes[2].fill_between(steps_a, policy_seq, step="post", alpha=0.5)
    axes[2].set_yticks([0, 1, 2])
    axes[2].set_yticklabels(policy_labels)
    axes[2].set_ylabel("active policy")
    axes[2].set_xlabel("request step")
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("matplotlib 不可用，文本摘要：")
    print("switch count:", len(adaptive.switch_log))
    print("final policy:", adaptive.current_policy)

## 12) Adaptive 策略面试讲解模板

> "线上服务的流量分布是动态变化的。仅靠静态 LRU 或 LFU，在热点漂移或租户负载突变时容易失效。
> 我的方案是：维护滑动窗口监控 hit rate 和 Jain fairness，每隔 N 步评估一次：
> - hit rate 低于阈值 → 切 LFU（频率信号更强）
> - fairness 低于阈值 → 切 Fair（配额驱逐保护小租户）
> - 两者都正常 → 回到 LRU（简单高效）
>
> 关键设计：冷却期防止振荡，窗口大小可按 SLO 调整。
> 回滚条件：若切换后窗口 hit rate 持续恶化两个周期，自动回退到 LRU baseline。"

### 面试追问准备

1. **窗口大小怎么选？** → 太小噪声大容易误判，太大响应滞后。建议 1-2 倍 eval_interval。
2. **冷却期怎么定？** → 至少覆盖一个完整窗口，否则还没看到新策略的效果就又切了。
3. **三种策略不够怎么办？** → 可以加"注意力感知"（H2O 式 score-based eviction）作为第四档。
4. **这个方案的缺点？** → 切换瞬间有短暂抖动；策略空间大时可能需要 bandit/RL 方法。